# Sensor Readings — Data Cleaning

- This notebook cleans and validates the raw sensor readings dataset for the Atmosyn project.

- The objective is to prepare a reliable dataset for further analysis while preserving important sensor events such as temperature excursions.

In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(r"C:\Users\VISHAL\OneDrive\Pictures\Documents\Infotact project\ATMOSYN_my\iot_sensor_readings.xls")
df.head()
# Loading the dataset

,reading_id,shipment_id,container_id,timestamp,temperature,humidity,vibration,latitude,longitude
0,READ0000001,SHP0001,CNT1001,2026-08-08 13:15:00,5.08,43.90,0.029,10.0317,76.2380
1,READ0000002,SHP0001,CNT1001,2026-08-08 13:30:00,5.17,40.51,0.153,10.1214,76.2015
2,READ0000003,SHP0001,CNT1001,2026-08-08 13:45:00,4.75,47.61,0.089,10.1998,76.1592
3,READ0000004,SHP0001,CNT1001,2026-08-08 14:00:00,4.71,47.30,0.093,10.2928,76.1340
4,READ0000005,SHP0001,CNT1001,2026-08-08 14:15:00,5.01,54.81,0.673,10.3941,76.0918


## 2. Create a Working Copy

In [4]:
cleaned_df = df.copy()

print("Working copy created.")

Working copy created.


## 3. Standardize Column Names

In [5]:
cleaned_df.columns = (
    cleaned_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(cleaned_df.columns.tolist())

['reading_id', 'shipment_id', 'container_id', 'timestamp', 'temperature', 'humidity', 'vibration', 'latitude', 'longitude']


## 4. Convert Data Types

In [6]:
cleaned_df["timestamp"] = pd.to_datetime(
    cleaned_df["timestamp"],
    errors="coerce"
)

numeric_columns = [
    "temperature",
    "humidity",
    "vibration",
    "latitude",
    "longitude"
]

for column in numeric_columns:
    cleaned_df[column] = pd.to_numeric(
        cleaned_df[column],
        errors="coerce"
    )

print(cleaned_df.dtypes)

reading_id              object
shipment_id             object
container_id            object
timestamp       datetime64[ns]
temperature            float64
humidity               float64
vibration              float64
latitude               float64
longitude              float64
dtype: object


In [7]:
print("Missing values after type conversion:")
print(cleaned_df.isnull().sum())

Missing values after type conversion:
reading_id      0
shipment_id     0
container_id    0
timestamp       0
temperature     0
humidity        0
vibration       0
latitude        0
longitude       0
dtype: int64


## 6. Remove Duplicate Records

In [8]:
before = len(cleaned_df)

cleaned_df = cleaned_df.drop_duplicates()

after = len(cleaned_df)

print("Duplicates removed:", before - after)
print("Current dataset shape:", cleaned_df.shape)

Duplicates removed: 0
Current dataset shape: (50000, 9)


## 7. Validate Sensor and Location Values

In [9]:
# Check for invalid humidity values
invalid_humidity = (
    (cleaned_df["humidity"] < 0) |
    (cleaned_df["humidity"] > 100)
).sum()

# Check for invalid latitude values
invalid_latitude = (
    (cleaned_df["latitude"] < -90) |
    (cleaned_df["latitude"] > 90)
).sum()

# Check for invalid longitude values
invalid_longitude = (
    (cleaned_df["longitude"] < -180) |
    (cleaned_df["longitude"] > 180)
).sum()

# Check for negative vibration values
negative_vibration = (
    cleaned_df["vibration"] < 0
).sum()

print("Invalid humidity values:", invalid_humidity)
print("Invalid latitude values:", invalid_latitude)
print("Invalid longitude values:", invalid_longitude)
print("Negative vibration values:", negative_vibration)

Invalid humidity values: 0
Invalid latitude values: 0
Invalid longitude values: 0
Negative vibration values: 0


## 8. Identify Temperature Excursions

In [10]:
cleaned_df["temperature_status"] = cleaned_df["temperature"].apply(
    lambda x: "Temperature Excursion" if x > 8 else "Normal"
)

print(cleaned_df["temperature_status"].value_counts())

temperature_status
Normal                   47106
Temperature Excursion     2894
Name: count, dtype: int64


In [11]:
high_temp_cleaned = cleaned_df[
    cleaned_df["temperature"] > 8
]

print("Temperature excursion records:", len(high_temp_cleaned))

display(high_temp_cleaned.head(10))

Temperature excursion records: 2894


,reading_id,shipment_id,container_id,timestamp,temperature,humidity,vibration,latitude,longitude,temperature_status
245,READ0000246,SHP0003,CNT1003,2026-08-19 06:45:00,8.02,60.20,0.148,9.1694,76.6253,Temperature Excursion
246,READ0000247,SHP0003,CNT1003,2026-08-19 07:09:00,8.16,53.61,0.546,9.1841,76.6284,Temperature Excursion
247,READ0000248,SHP0003,CNT1003,2026-08-19 07:33:00,8.44,49.71,0.058,9.1968,76.6133,Temperature Excursion
248,READ0000249,SHP0003,CNT1003,2026-08-19 07:57:00,8.58,55.09,0.119,9.2110,76.6032,Temperature Excursion
249,READ0000250,SHP0003,CNT1003,2026-08-19 08:21:00,8.90,52.98,0.024,9.2269,76.5995,Temperature Excursion
250,READ0000251,SHP0003,CNT1003,2026-08-19 08:45:00,9.11,54.50,0.117,9.2385,76.5929,Temperature Excursion
251,READ0000252,SHP0003,CNT1003,2026-08-19 09:09:00,9.40,63.89,0.488,9.2611,76.5948,Temperature Excursion
252,READ0000253,SHP0003,CNT1003,2026-08-19 09:33:00,9.72,46.09,0.030,9.2600,76.5848,Temperature Excursion
253,READ0000254,SHP0003,CNT1003,2026-08-19 09:57:00,9.87,52.15,0.066,9.2926,76.5722,Temperature Excursion
254,READ0000255,SHP0003,CNT1003,2026-08-19 10:21:00,10.11,47.78,0.135,9.2942,76.5745,Temperature Excursion


## 9. Validate Timestamp Data

In [12]:
print("Invalid timestamps:",
      cleaned_df["timestamp"].isna().sum())

print("Earliest timestamp:",
      cleaned_df["timestamp"].min())

print("Latest timestamp:",
      cleaned_df["timestamp"].max())

Invalid timestamps: 0
Earliest timestamp: 2026-08-01 06:30:00
Latest timestamp: 2026-08-28 08:21:45


In [13]:
# Check whether timestamps are in chronological order
is_sorted = cleaned_df["timestamp"].is_monotonic_increasing

print("Timestamps globally sorted:", is_sorted)

Timestamps globally sorted: False


## 10. Sort the Cleaned Dataset

In [14]:
cleaned_df = cleaned_df.sort_values(
    by=["shipment_id", "container_id", "timestamp"]
).reset_index(drop=True)

print("Dataset sorted successfully.")

display(cleaned_df.head(10))

Dataset sorted successfully.


,reading_id,shipment_id,container_id,timestamp,temperature,humidity,vibration,latitude,longitude,temperature_status
0,READ0000001,SHP0001,CNT1001,2026-08-08 13:15:00,5.08,43.90,0.029,10.0317,76.2380,Normal
1,READ0000002,SHP0001,CNT1001,2026-08-08 13:30:00,5.17,40.51,0.153,10.1214,76.2015,Normal
2,READ0000003,SHP0001,CNT1001,2026-08-08 13:45:00,4.75,47.61,0.089,10.1998,76.1592,Normal
3,READ0000004,SHP0001,CNT1001,2026-08-08 14:00:00,4.71,47.30,0.093,10.2928,76.1340,Normal
4,READ0000005,SHP0001,CNT1001,2026-08-08 14:15:00,5.01,54.81,0.673,10.3941,76.0918,Normal
5,READ0000006,SHP0001,CNT1001,2026-08-08 14:30:00,5.36,64.14,0.069,10.4733,76.0552,Normal
6,READ0000007,SHP0001,CNT1001,2026-08-08 14:45:00,4.95,43.05,0.026,10.5633,76.0337,Normal
7,READ0000008,SHP0001,CNT1001,2026-08-08 15:00:00,5.13,47.79,0.107,10.6710,75.9913,Normal
8,READ0000009,SHP0001,CNT1001,2026-08-08 15:15:00,5.22,63.49,0.116,10.7479,75.9716,Normal
9,READ0000010,SHP0001,CNT1001,2026-08-08 15:30:00,4.76,41.13,0.082,10.8541,75.9201,Normal


### Verify the sorting

In [15]:
print(
    "First shipment:",
    cleaned_df["shipment_id"].iloc[0]
)

print(
    "First container:",
    cleaned_df["container_id"].iloc[0]
)

print(
    "First timestamp:",
    cleaned_df["timestamp"].iloc[0]
)

First shipment: SHP0001
First container: CNT1001
First timestamp: 2026-08-08 13:15:00


## 11. Final Data Quality Check

In [16]:
print("Final Dataset Shape:", cleaned_df.shape)

print("\nMissing Values:")
print(cleaned_df.isnull().sum())

print("\nDuplicate Rows:")
print(cleaned_df.duplicated().sum())

print("\nData Types:")
print(cleaned_df.dtypes)

Final Dataset Shape: (50000, 10)

Missing Values:
reading_id            0
shipment_id           0
container_id          0
timestamp             0
temperature           0
humidity              0
vibration             0
latitude              0
longitude             0
temperature_status    0
dtype: int64

Duplicate Rows:
0

Data Types:
reading_id                    object
shipment_id                   object
container_id                  object
timestamp             datetime64[ns]
temperature                  float64
humidity                     float64
vibration                    float64
latitude                     float64
longitude                    float64
temperature_status            object
dtype: object


In [17]:
print("\nTemperature Status:")
print(cleaned_df["temperature_status"].value_counts())


Temperature Status:
temperature_status
Normal                   47106
Temperature Excursion     2894
Name: count, dtype: int64


## 12. Export Cleaned Dataset

In [19]:
output_path = "sensor_readings_cleaned.csv"

cleaned_df.to_csv(
    output_path,
    index=False
)

print("Cleaned dataset exported successfully.")
print("File:", output_path)

Cleaned dataset exported successfully.
File: sensor_readings_cleaned.csv


In [20]:
import os

print("File exists:", os.path.exists(output_path))

File exists: True


## 13. Final Verification of Cleaning Steps

The following checks verify that all data-cleaning operations were completed successfully:

1. The raw dataset was loaded without modifying the original file.
2. A working copy was created for cleaning.
3. Column names were standardized.
4. Data types were converted to appropriate formats.
5. Missing values were checked after conversion.
6. Duplicate records were checked and removed if present.
7. Sensor and geographical values were validated.
8. Temperature excursions above 8°C were identified and classified.
9. Timestamp values were validated.
10. The dataset was sorted by shipment, container, and timestamp.
11. Final data quality checks were performed.
12. The cleaned dataset was exported as `sensor_readings_cleaned.csv`.

# FINAL VERIFICATION

In [21]:
print("========== FINAL VERIFICATION ==========")

print("\n1. Dataset Shape:")
print(cleaned_df.shape)

print("\n2. Missing Values:")
print(cleaned_df.isnull().sum().sum())

print("\n3. Duplicate Rows:")
print(cleaned_df.duplicated().sum())

print("\n4. Temperature Status:")
print(cleaned_df["temperature_status"].value_counts())

print("\n5. Data Types:")
print(cleaned_df.dtypes)

print("\n6. Exported File:")
print(os.path.exists(output_path))

print("\n========== VERIFICATION COMPLETE ==========")

========== FINAL VERIFICATION ==========

1. Dataset Shape:
(50000, 10)

2. Missing Values:
0

3. Duplicate Rows:
0

4. Temperature Status:
temperature_status
Normal                   47106
Temperature Excursion     2894
Name: count, dtype: int64

5. Data Types:
reading_id                    object
shipment_id                   object
container_id                  object
timestamp             datetime64[ns]
temperature                  float64
humidity                     float64
vibration                    float64
latitude                     float64
longitude                    float64
temperature_status            object
dtype: object

6. Exported File:
True

========== VERIFICATION COMPLETE ==========
